Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
#@title Imports (run this first)
from fractions import Fraction
import itertools
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Packing circles inside a unit square to maximize sum of radii



Given a positive integer $n$, the problem is to pack $n$ disjoint circles inside a unit square so as to maximize the sum of their radii. The state of the art can be found on [Erich Friedman's homepage](https://erich-friedman.github.io/packing/cirRsqu/).

* For $n=26$, the SOTA was $2.634$, and AlphaEvolve improved it to $2.635$ (Construction 1).
* For $n=32$, the SOTA was $2.936$, and AlphaEvolve improved it to $2.939$ (Construction 2).

The constructions found by AlphaEvolve are shown below.

In [ ]:
#@title Verification function

def verify_circles_exact(
    points: np.ndarray,
    radii: np.ndarray,
    dims: tuple[float, float],
) -> tuple[bool, float]:
  """Verifies the circle packing using exact integer arithmetic.

  Converts all inputs to Fractions and scales them by the LCM of all
  denominators to perform calculations using arbitrary-precision integers.

  Args:
    points: (n, 2) array of circle centers.
    radii: (n,) array of circle radii.
    dims: (width, height) of the bounding rectangle.

  Returns:
    A tuple (is_valid, sum_radii):
    If valid packing: (True, sum of radii rounded down to nearest float).
    If invalid packing: (False, 0.0).
  """
  # 1. Convert everything to exact Fractions.
  w_f = Fraction(dims[0])
  h_f = Fraction(dims[1])
  f_pts = [(Fraction(p[0]), Fraction(p[1])) for p in points]
  f_radii = [Fraction(r) for r in radii]

  # 2. Find global LCM to scale to integers.
  all_denoms = {w_f.denominator, h_f.denominator}
  for x, y in f_pts:
    all_denoms.add(x.denominator)
    all_denoms.add(y.denominator)
  for r in f_radii:
    all_denoms.add(r.denominator)

  multiplier = math.lcm(*all_denoms)

  def scale_to_int(f):
    return f.numerator * (multiplier // f.denominator)

  # 3. Scale all parameters to Python integers.
  W_i, H_i = scale_to_int(w_f), scale_to_int(h_f)
  pts_i = [(scale_to_int(p[0]), scale_to_int(p[1])) for p in f_pts]
  rad_i = [scale_to_int(r) for r in f_radii]

  # 4. Global Constraint Check (w + h <= 2)
  if W_i + H_i > 2 * multiplier:
    print(f"Validation Failed: sum of dimensions {dims[0] + dims[1]} > 2.0")
    return False, 0.0

  # 5. Loop over all circles
  n = len(pts_i)
  for i in range(n):
    xi, yi = pts_i[i]
    ri = rad_i[i]

    # Boundary Check
    if xi - ri < 0 or xi + ri > W_i or yi - ri < 0 or yi + ri > H_i:
      print(f"Validation Failed: Circle {i} at {points[i]} with radius {radii[i]} is out of bounds.")
      return False, 0.0

    # Overlap Check
    for j in range(i + 1, n):
      xj, yj = pts_i[j]
      rj = rad_i[j]

      # (dx^2 + dy^2) < (r1 + r2)^2
      dist_sq = (xi - xj)**2 + (yi - yj)**2
      sum_r_sq = (ri + rj)**2

      if dist_sq < sum_r_sq:
        print(f"Validation Failed: Circle {i} and Circle {j} overlap.")
        return False, 0.0

  # 6. Success: Calculate radii sum and convert to float (round down)
  total_radii_sum = Fraction(sum(rad_i), multiplier)

  result = float(total_radii_sum)
  if Fraction(result) > total_radii_sum:
    result = math.nextafter(result, -math.inf)

  assert Fraction(result) <= total_radii_sum, "Rounding error"

  return True, result

In [ ]:
#@title Visualization function

def visualize_packing(
    centers: np.ndarray,
    radii: np.ndarray,
    width: float,
    height: float,
    title: str = "Circle Packing"
):
    """
    Creates and displays a plot of the circle packing within a rectangle.

    Args:
        centers: An (n, 2) array of circle centers.
        radii: An (n,) array of circle radii.
        width: The width of the rectangle.
        height: The height of the rectangle.
        title: An optional title for the plot.
    """
    _, ax = plt.subplots(1, figsize=(8, 8))

    # 1. Set the plot to have an equal aspect ratio
    ax.set_aspect('equal', adjustable='box')

    # 2. Draw the rectangle container
    rect = patches.Rectangle(
        (0, 0),
        width,
        height,
        linewidth=2,
        edgecolor='black',
        facecolor='none',
        label='Boundary'
    )
    ax.add_patch(rect)

    # 3. Draw each circle and label it with its index
    for i in range(len(radii)):
        circle = patches.Circle(
            (centers[i, 0], centers[i, 1]),
            radii[i],
            edgecolor='royalblue',
            facecolor='skyblue',
            alpha=0.6
        )
        ax.add_patch(circle)
        # Add the index number 'i' to the center of the circle
        ax.text(centers[i, 0], centers[i, 1], str(i),
                ha='center', va='center', fontsize=8, color='navy')

    # 4. Set plot limits and appearance
    margin = max(width, height) * 0.05  # 5% margin
    ax.set_xlim(-margin, width + margin)
    ax.set_ylim(-margin, height + margin)
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Width")
    ax.set_ylabel("Height")

    # 5. Show the plot
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()


In [ ]:
#@title Construction 1: Data
centers_26 = np.array([[0.7026096036990247 , 0.3816658444526457 ], [0.31311580997040145, 0.9076084484290411 ], [0.726905714311596 , 0.5960427019081348 ], [0.894817439731251 , 0.27395283962395217], [0.9038486659542352 , 0.6820800429325902 ], [0.10306052014158237, 0.48460080265191496], [0.2716298514860014 , 0.5976347963876618 ], [0.23971052792753603, 0.7636735693833854 ], [0.759352401560932 , 0.7629588636539982 ], [0.915073737545101 , 0.084926262454899 ], [0.4955317606743583 , 0.2753426167714399 ], [0.896532766642048 , 0.48259558221054216], [0.49728444620411005, 0.07886037291596384], [0.11077901279071568, 0.8892209872092842 ], [0.8888438205895552 , 0.8888438205895551 ], [0.49942836913903915, 0.9060726627225563 ], [0.4986680755034023 , 0.5299634197531913 ], [0.08463950069577313, 0.08463950069577317], [0.2946094887818273 , 0.1302211010652256 ], [0.4033587836026808 , 0.742417049501635 ], [0.6859430219867827 , 0.9074079050485644 ], [0.7023095250891603 , 0.13325857277081155], [0.5952197329397327 , 0.742049443460592 ], [0.09573232930702141, 0.6832585349730799 ], [0.29474605904894957, 0.38692355340959994], [0.10679014462858022, 0.2747832833508202 ]])
radii_26 = np.array([0.11514878016002295, 0.09239145157095896, 0.1006003678187114 , 0.105182460268749  , 0.09615123404576487, 0.10306042014158237, 0.09989835059275479, 0.06918067635723468, 0.0694401937112584 , 0.08492616245489899, 0.11762968804654092, 0.1034671333579521 , 0.07886027291596384, 0.11077891279071568, 0.11115607941044481, 0.09392723727744379, 0.13701038012374772, 0.08463940069577312, 0.13022100106522516, 0.09584227574550488, 0.09259199495143566, 0.13325847277081154, 0.09601892575825385, 0.09573222930702141, 0.1120769889502572 , 0.10679004462858022])

In [ ]:
#@title Construction 1: Verification
print(f"Construction 1 has {len(centers_26)} circles.")
is_valid, result = verify_circles_exact(centers_26, radii_26, (1, 1))
if is_valid:
  print(f"Construction 1 is valid and has sum of radii: {result}")
else:
  print("Construction 1 is invalid.")


In [ ]:
#@title Construction 1: Visualization
visualize_packing(centers_26, radii_26, 1, 1, "AlphaEvolve's 26-Circle Packing")

In [ ]:
#@title Construction 2: Data

centers_32 = np.array([[0.4103367574347774 , 0.4034775285334993 ], [0.8963458552602112 , 0.10365414473978883], [0.2466174786966004 , 0.6773614002482867 ], [0.40962670073476165, 0.21822588350952354], [0.24333518337797744, 0.1079405010782716 ], [0.07306696147175942, 0.9269330385282406 ], [0.09052877125190313, 0.5853168363356677 ], [0.5726977383625569 , 0.106228715042693 ], [0.7369697857091396 , 0.21625157750836602], [0.573911802908532 , 0.3081330707478824 ], [0.4086745937675105 , 0.06331525320633652], [0.7355143578272628 , 0.06238720755320459], [0.25728850226113253, 0.8838819371737265 ], [0.8884332917756496 , 0.8884332917756496 ], [0.739436872889558 , 0.4026491108074175 ], [0.572899637814724 , 0.7133709992493075 ], [0.24844240778977586, 0.3094037922636657 ], [0.5743914194083511 , 0.5010189546480183 ], [0.09056937488828318, 0.4042186947473096 ], [0.9061987946708536 , 0.6838352301421212 ], [0.0875712557500808 , 0.2261032951720141 ], [0.9034277921600417 , 0.30375521529448646], [0.3904454681243297 , 0.7519411666029409 ], [0.4712803348377272 , 0.9014096013683925 ], [0.7562770574133956 , 0.75857067081745 ], [0.08942465018367278, 0.7652668705202621 ], [0.06977107568297637, 0.06977107568297637], [0.7427641440618451 , 0.5914750415187353 ], [0.9051413826149427 , 0.4951783707749125 ], [0.2488889588844859 , 0.494844162673007 ], [0.6665042606822044 , 0.90335676254708 ], [0.4068368918378875 , 0.5892639181130421 ]])
radii_32 = np.array([0.09365470284345939 , 0.10365414473978873 , 0.09067797900211055 , 0.09159830297454696 , 0.10794050107827151 , 0.07306696147175933 , 0.09052877125190303 , 0.1062287150426929  , 0.09148404581147726 , 0.09567929075561449 , 0.06331525320633642 , 0.062387207553204496, 0.11611806282627335 , 0.11156670822435033 , 0.09492981348854032 , 0.1151500950153154  , 0.09358751542673782 , 0.09720718943396117 , 0.09056937488828308 , 0.09380120532914626 , 0.0875712557500807  , 0.09657220783995824 , 0.07133631694839429 , 0.09859039863160735 , 0.07371569947520412 , 0.08942465018367268 , 0.06977107568297627 , 0.09392542960110564 , 0.09485861738505719 , 0.09185339264220435 , 0.09664323745291992 , 0.09216464924936738 ])

In [ ]:
#@title Construction 2: Verification
print(f"Construction 2 has {len(centers_32)} circles.")
is_valid, result = verify_circles_exact(centers_32, radii_32, (1, 1))
if is_valid:
  print(f"Construction 2 is valid and has sum of radii: {result}")
else:
  print("Construction 2 is invalid.")

In [ ]:
#@title Construction 2: Visualization
visualize_packing(centers_32, radii_32, 1, 1, "AlphaEvolve's 32-Circle Packing")

# Packing circles inside a rectangle of perimeter 4 to maximize sum of radii

Given a positive integer $n$, the problem is to pack $n$ disjoint circles inside a rectangle of perimeter 4 so as to maximize the sum of their radii. The state of the art can be found on [Erich Friedman's homepage](https://erich-friedman.github.io/packing/cirRrec/).

For $n=21$, the SOTA was $2.364$, and AlphaEvolve improved it to $2.365$. The construction found by AlphaEvolve is shown below.

In [ ]:
#@title Construction 3: Data
centers_21 = np.array([[0.5116344482349532 , 0.6042884846390946 ], [0.7087426256542229 , 0.7175624677635186 ], [0.3145262708156835 , 0.7175624677635184 ], [0.31647214265648654, 0.9019896034582606 ], [0.899307288731929 , 0.8527694957921154 ], [0.7141245569697637 , 0.4958995816816336 ], [0.29193030461281383, 0.29820323112068375], [0.11292169820594489, 0.3858951194945234 ], [0.30914433950014253, 0.49589958168163345], [0.6306973045788548 , 0.11906285634390201], [0.7313385918570933 , 0.29820323112068287], [0.9081857702350873 , 0.6138896988097773 ], [0.7067967538134197 , 0.9019896034582613 ], [0.1370712971165672 , 0.13707129711656757], [0.9103471982639619 , 0.3858951194945241 ], [0.1239616077379781 , 0.8527694957921159 ], [0.5116344482349533 , 0.8493309125983606 ], [0.5116344482349529 , 0.34782102271875326], [0.39257159189105156, 0.11906285634390204], [0.8861975993533394 , 0.13707129711656638], [0.11508312623481891, 0.6138896988097775 ]])
radii_21 = np.array([0.1176421870275316 , 0.10969585068459667, 0.10969580068459697, 0.07474135007183358, 0.1239615577379769 , 0.11203226188844492, 0.08641206324529366, 0.11292159820594488, 0.11203231188844459, 0.11906275634390201, 0.08641201324529507, 0.11508307623481862, 0.07474135007183291, 0.1370711971165672 , 0.11292164820594391, 0.1239615077379781 , 0.12740004093173363, 0.1388251248928096 , 0.11906275634390204, 0.13707119711656637, 0.1150830262348189 ])
width, height = (1.0232689464699058, 0.9767310535300942)

In [ ]:
#@title Construction 3: Verification
print(f"Construction 3 has {len(centers_21)} circles.")
is_valid, result = verify_circles_exact(centers_21, radii_21, (width, height))
if is_valid:
  print(f"Construction 3 is valid and has sum of radii: {result} with {width=} and {height=}")
else:
  print("Construction 3 is invalid.")

In [ ]:
#@title Visualization
visualize_packing(centers_21, radii_21, width, height, "AlphaEvolve's 21-Circle Packing")